# Đào tạo Mô hình Dự đoán Biến chứng Tim mạch
Dựa trên dữ liệu đã được tiền xử lý và báo cáo EDA, notebook này sẽ thực hiện huấn luyện các mô hình Machine Learning: Logistic Regression, Random Forest và XGBoost.
Mục tiêu chính là tối ưu hóa chỉ số **Recall** để hạn chế bỏ sót bệnh nhân có nguy cơ cao.

In [5]:
import sys
import os
import warnings
import joblib
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    brier_score_loss, classification_report,
)
from sklearn.calibration     import calibration_curve
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"font.family": "DejaVu Sans"})

## 0. Thiết lập các hằng số, đường dẫn và hàm hỗ trợ vẽ biểu đồ
Định nghĩa các đường dẫn thư mục, màu sắc, và các hàm hỗ trợ trực quan hóa.

In [7]:
# =============================================================================
# 0. PATHS & DIRECTORIES
# =============================================================================
PROCESSED_DIR = os.path.join("data", "processed")
STAT_CSV      = os.path.join("outputs", "statistical_summary.csv")
MODEL_DIR     = os.path.join("models")
RESULT_DIR    = os.path.join("outputs", "model_results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# ── Visual style ──────────────────────────────────────────────────────────
BG       = "#FAFBFD"
GRID_CLR = "#E4E8ED"
PALETTE  = {"Logistic Regression": "#4A90D9",
            "Random Forest":       "#27AE60",
            "XGBoost":             "#E67E22"}

def style_ax(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(BG)
    ax.grid(color=GRID_CLR, linewidth=0.7, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#C0C4CA")
    if title:  ax.set_title(title, fontweight="bold", pad=8)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)

def save_fig(fig, name):
    path = os.path.join(RESULT_DIR, name)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  [Saved] {path}")

## 1. Tải dữ liệu đã qua tiền xử lý
Đọc dữ liệu huấn luyện và kiểm thử (đã qua SMOTE, loại bỏ ngoại lai và Scale) từ thư mục `data/processed/`.

In [9]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("\n" + "=" * 70)
print("  STEP 1 — LOADING PREPROCESSED DATA")
print("=" * 70)

X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train_final.csv"))
X_test  = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test_final.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train_final.csv")).squeeze()
y_test  = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test_final.csv")).squeeze()

print(f"  X_train : {X_train.shape}  |  y_train distribution: {dict(y_train.value_counts())}")
print(f"  X_test  : {X_test.shape}   |  y_test  distribution: {dict(y_test.value_counts())}")


  STEP 1 — LOADING PREPROCESSED DATA
  X_train : (18754, 30)  |  y_train distribution: {0: np.int64(9377), 1: np.int64(9377)}
  X_test  : (2471, 30)   |  y_test  distribution: {0: np.int64(2344), 1: np.int64(127)}


## 2. Lựa chọn đặc trưng (Feature Selection)
Dựa vào báo cáo thống kê (`statistical_summary.csv`), loại bỏ các biến không có ý nghĩa thống kê (p-value >= 0.05). Lưu lại danh sách các biến được giữ lại để sử dụng cho dự đoán.

In [10]:
# =============================================================================
# 2. FEATURE SELECTION  (based on EDA statistical summary)
# =============================================================================
print("\n" + "=" * 70)
print("  STEP 2 — FEATURE SELECTION (EDA-guided)")
print("=" * 70)

# Những biến này không cho thấy sự khác biệt có ý nghĩa thống kê giữa nhóm có và không có biến chứng
stat_df = pd.read_csv(STAT_CSV)

# Lấy danh sách biến không có ý nghĩa thống kê
nonsig_cols = stat_df.loc[~stat_df["Significant"], "OriginalCol"].tolist()
# Tang_huyet_ap không có biến động (variance = 0), bắt buộc loại bỏ
if "Tang_huyet_ap" not in nonsig_cols:
    nonsig_cols.append("Tang_huyet_ap")

# Chỉ loại bỏ những cột thực sự tồn tại trong dataset
cols_to_drop = [c for c in nonsig_cols if c in X_train.columns]

print(f"\nĐang loại bỏ {len(cols_to_drop)} biến không quan trọng / zero-variance:")
for c in cols_to_drop:
    row = stat_df[stat_df["OriginalCol"] == c]
    pval = f"p={row['p-value'].values[0]:.4f}" if not row.empty else "zero-variance"
    print(f"    [-] {c:<25}  ({pval})")

X_train_sel = X_train.drop(columns=cols_to_drop)
X_test_sel  = X_test.drop(columns=cols_to_drop)

print(f"\nSố lượng đặc trưng ban đầu : {X_train.shape[1]}")
print(f"\nSố lượng sau khi chọn lọc  : {X_train_sel.shape[1]}")

# Lưu danh sách biến này lại cho việc dự đoán (như Streamlit App)
selected_features = X_train_sel.columns.tolist()
feature_list_path = os.path.join(MODEL_DIR, "selected_features.json")
with open(feature_list_path, "w", encoding="utf-8") as f:
    json.dump(selected_features, f, indent=2)
print(f"\nDanh sách đặc trưng đã lưu -> {feature_list_path}")


  STEP 2 — FEATURE SELECTION (EDA-guided)

Đang loại bỏ 10 biến không quan trọng / zero-variance:
    [-] HBa1C_mean                 (p=0.6205)
    [-] HBa1C_max                  (p=0.8192)
    [-] HBa1C_min                  (p=0.5363)
    [-] HBa1C_std                  (p=0.8882)
    [-] HBa1C_last                 (p=0.4889)
    [-] HDL_max                    (p=0.2930)
    [-] Triglycerid_max            (p=0.0613)
    [-] Triglycerid_std            (p=0.9634)
    [-] benh_mach_vanh             (p=1.0000)
    [-] Tang_huyet_ap              (zero-variance)

Số lượng đặc trưng ban đầu : 30

Số lượng sau khi chọn lọc  : 20

Danh sách đặc trưng đã lưu -> models\selected_features.json


## 3. Khởi tạo mô hình và không gian siêu tham số (Hyperparameter Grids)
Khởi tạo các mô hình và tham số cho GridSearchCV: **Logistic Regression**, **Random Forest**, **XGBoost**. Thiết lập metric đánh giá chính là **Recall** (Sensitivity) để giảm thiểu bỏ sót bệnh.

In [11]:
# =============================================================================
# 3. MODEL DEFINITIONS + HYPERPARAMETER GRIDS
# =============================================================================
print("\n" + "=" * 70)
print("  STEP 3 — MODEL DEFINITIONS & HYPERPARAMETER GRIDS")
print("=" * 70)

# Sử dụng Stratified K-Fold để đảm bảo tỷ lệ lớp mục tiêu được duy trì trong từng fold.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Chỉ số tối ưu chính: Recall (Sensitivity)
SCORING = "recall"

# ── Mô hình 1: Logistic Regression ─────────────────────────────────────────
lr_model  = LogisticRegression(penalty="l2", solver="liblinear",
                               class_weight="balanced", max_iter=1000, random_state=42)
lr_params = {"C": [0.001, 0.01, 0.1, 1, 10, 100]}

# ── Mô hình 2: Random Forest ────────────────────────────────────────────────
rf_model  = RandomForestClassifier(class_weight="balanced_subsample",
                                   random_state=42, n_jobs=-1)
rf_params = {
    "n_estimators":    [100, 200, 300],
    "max_depth":       [5, 10, 15],
    "min_samples_split": [2, 5, 10],
}

# ── Mô hình 3: XGBoost ─────────────────────────────────────────────────────
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
spw   = round(n_neg / n_pos, 2)
print(f"\nXGBoost scale_pos_weight = {n_neg}/{n_pos} = {spw}")

xgb_model  = XGBClassifier(scale_pos_weight=spw, eval_metric="logloss",
                            random_state=42, n_jobs=-1)
xgb_params = {
    "learning_rate":    [0.01, 0.1, 0.2],
    "max_depth":        [3, 5, 7],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

MODELS = {
    "Logistic Regression": (lr_model,  lr_params),
    "Random Forest":       (rf_model,  rf_params),
    "XGBoost":             (xgb_model, xgb_params),
}


  STEP 3 — MODEL DEFINITIONS & HYPERPARAMETER GRIDS

XGBoost scale_pos_weight = 9377/9377 = 1.0


## 4. Huấn luyện bằng GridSearchCV (5-Fold Stratified CV)
Huấn luyện các mô hình và tìm siêu tham số tối ưu bằng GridSearchCV. Lưu lại các mô hình tốt nhất vào thư mục `models/`.

In [12]:
# =============================================================================
# 4. TRAINING WITH GridSearchCV (5-Fold Stratified CV)
# =============================================================================
print("\n" + "=" * 70)
print("  STEP 4 — GRIDSEARCHCV TRAINING (5-Fold Stratified CV)")
print("=" * 70)

best_estimators = {}   # Lưu trữ mô hình tốt nhất
cv_results_log  = []   # Lưu trữ tóm tắt quá trình huấn luyện

for model_name, (base_model, param_grid) in MODELS.items():
    print(f"\n[ {model_name} ]")
    print(f"\nGrid size : {np.prod([len(v) for v in param_grid.values()])} combinations")

    gs = GridSearchCV(
        estimator  = base_model,
        param_grid = param_grid,
        cv         = CV,
        scoring    = SCORING,          # Optimize for Recall
        refit      = True,             # Refit trên toàn bộ tập train
        n_jobs     = -1,
        verbose    = 1,
    )
    gs.fit(X_train_sel, y_train)

    best_estimators[model_name] = gs.best_estimator_
    best_recall_cv = gs.best_score_

    print(f"    Best params       : {gs.best_params_}")
    print(f"    Best CV Recall    : {best_recall_cv:.4f}")

    cv_results_log.append({
        "Model":           model_name,
        "Best_Params":     str(gs.best_params_),
        "CV_Recall_Mean":  round(best_recall_cv, 4),
    })

    # ── Save best model to models/ directory ─────────────────────────────
    safe_name   = model_name.lower().replace(" ", "_")
    model_path  = os.path.join(MODEL_DIR, f"best_{safe_name}.pkl")
    joblib.dump(gs.best_estimator_, model_path)
    print(f"    Saved model       : {model_path}")


  STEP 4 — GRIDSEARCHCV TRAINING (5-Fold Stratified CV)

[ Logistic Regression ]

Grid size : 6 combinations
Fitting 5 folds for each of 6 candidates, totalling 30 fits
    Best params       : {'C': 0.001}
    Best CV Recall    : 0.7900
    Saved model       : models\best_logistic_regression.pkl

[ Random Forest ]

Grid size : 27 combinations
Fitting 5 folds for each of 27 candidates, totalling 135 fits
    Best params       : {'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 300}
    Best CV Recall    : 0.9195
    Saved model       : models\best_random_forest.pkl

[ XGBoost ]

Grid size : 36 combinations
Fitting 5 folds for each of 36 candidates, totalling 180 fits
    Best params       : {'colsample_bytree': 0.9, 'learning_rate': 0.2, 'max_depth': 7, 'subsample': 0.7}
    Best CV Recall    : 0.9654
    Saved model       : models\best_xgboost.pkl


## 5. Đánh giá trên tập dữ liệu Test (Hold-out)
Tính toán các chỉ số: Accuracy, Precision, Recall, F1-Score, ROC-AUC, Brier score trên tập Test.

In [13]:
# =============================================================================
# 5. EVALUATION ON TEST SET
# =============================================================================
print("\n" + "=" * 70)
print("  STEP 5 — EVALUATION ON HELD-OUT TEST SET")
print("=" * 70)

metrics_rows = []
roc_data     = {}   # For ROC curve plot

for model_name, estimator in best_estimators.items():
    y_pred      = estimator.predict(X_test_sel)
    y_prob      = estimator.predict_proba(X_test_sel)[:, 1]

    acc     = accuracy_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred, zero_division=0)
    rec     = recall_score(y_test, y_pred)
    f1      = f1_score(y_test, y_pred, zero_division=0)
    auc     = roc_auc_score(y_test, y_prob)
    brier   = brier_score_loss(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    roc_data[model_name] = (fpr, tpr, auc)

    print(f"\n  [ {model_name} ]")
    print(f"\n  Accuracy  : {acc:.4f}")
    print(f"\n  Precision : {prec:.4f}")
    print(f"\n  Recall    : {rec:.4f}  <-- PRIMARY METRIC")
    print(f"\n  F1-Score  : {f1:.4f}")
    print(f"\n  ROC-AUC   : {auc:.4f}")
    print(f"\n  Brier     : {brier:.4f}")
    print(f"\n  Classification Report: {classification_report(y_test, y_pred, target_names=['No Complication', 'Complication'])}")

    metrics_rows.append({
        "Model":     model_name,
        "Accuracy":  round(acc,   4),
        "Precision": round(prec,  4),
        "Recall":    round(rec,   4),
        "F1-Score":  round(f1,    4),
        "ROC-AUC":   round(auc,   4),
        "Brier":     round(brier, 4),
    })

# ── Bảng so sánh các chỉ số ────────────────────────────────────────
metrics_df = pd.DataFrame(metrics_rows).sort_values(
    by=["Recall", "F1-Score", "ROC-AUC"], 
    ascending=[False, False, False]
)
metrics_csv = os.path.join(RESULT_DIR, "model_comparison.csv")
metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
print(f"\n[Saved] Metrics table -> {metrics_csv}\n")
display(metrics_df)


  STEP 5 — EVALUATION ON HELD-OUT TEST SET

  [ Logistic Regression ]

  Accuracy  : 0.8693

  Precision : 0.2621

  Recall    : 0.8504  <-- PRIMARY METRIC

  F1-Score  : 0.4007

  ROC-AUC   : 0.9414

  Brier     : 0.1439

  Classification Report:                  precision    recall  f1-score   support

No Complication       0.99      0.87      0.93      2344
   Complication       0.26      0.85      0.40       127

       accuracy                           0.87      2471
      macro avg       0.63      0.86      0.66      2471
   weighted avg       0.95      0.87      0.90      2471


  [ Random Forest ]

  Accuracy  : 0.9895

  Precision : 0.8531

  Recall    : 0.9606  <-- PRIMARY METRIC

  F1-Score  : 0.9037

  ROC-AUC   : 0.9831

  Brier     : 0.0326

  Classification Report:                  precision    recall  f1-score   support

No Complication       1.00      0.99      0.99      2344
   Complication       0.85      0.96      0.90       127

       accuracy                   

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Brier
2,XGBoost,0.9939,0.9242,0.9606,0.9421,0.9807,0.0109
1,Random Forest,0.9895,0.8531,0.9606,0.9037,0.9831,0.0326
0,Logistic Regression,0.8693,0.2621,0.8504,0.4007,0.9414,0.1439


## 6. Chọn Mô hình Tốt nhất (Champion Model)
Lựa chọn mô hình tốt nhất dựa trên tiêu chí Recall cao nhất. Lưu thông tin meta để tham khảo cho các ứng dụng Demo (như Streamlit).

In [14]:
# =============================================================================
# 6. BEST MODEL SELECTION & ADDITIONAL SAVE
# =============================================================================
print("\n" + "=" * 70)
print("  STEP 6 — BEST MODEL SELECTION")
print("=" * 70)

# Tiêu chí chọn lựa (Primary): Recall cao nhất. Tie-break: AUC rồi F1.
best_row   = metrics_df.iloc[0]
best_name  = best_row["Model"]
best_model = best_estimators[best_name]

print(f"\n  Mô hình tốt nhất (Champion): {best_name}")
print(f"\n  Recall                     : {best_row['Recall']:.4f}")
print(f"\n  AUC                        : {best_row['ROC-AUC']:.4f}")
print(f"\n  F1-Score                   : {best_row['F1-Score']:.4f}")

# Save champion model
champion_path = os.path.join(MODEL_DIR, "champion_model.pkl")
joblib.dump(best_model, champion_path)
print(f"\n  Champion model saved -> {champion_path}")

meta = {"champion": best_name, "metrics": best_row.to_dict()}
with open(os.path.join(MODEL_DIR, "champion_meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)


  STEP 6 — BEST MODEL SELECTION

  Mô hình tốt nhất (Champion): XGBoost

  Recall                     : 0.9606

  AUC                        : 0.9807

  F1-Score                   : 0.9421

  Champion model saved -> models\champion_model.pkl


## 7. Trực quan hóa (Visualizations)
Vẽ các biểu đồ: Confusion Matrix, ROC Curve, Calibration Curve, và Feature Importances.\n

In [15]:
# =============================================================================
# 7. VISUALIZATION
# =============================================================================
print("\n" + "=" * 70)
print("\n  STEP 7 — GENERATING EVALUATION PLOTS")
print("\n" + "=" * 70)

# ── 7a. Confusion Matrices (tất cả mô hình) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle("Confusion Matrix — All Models (Test Set)", fontsize=14, fontweight="bold")

for ax, (model_name, estimator) in zip(axes, best_estimators.items()):
    y_pred = estimator.predict(X_test_sel)
    cm     = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Comp.", "Comp."],
                yticklabels=["No Comp.", "Comp."],
                linewidths=0.5, linecolor="#DDD",
                annot_kws={"size": 13, "weight": "bold"})
    ax.set_title(model_name, fontweight="bold", pad=8)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

fig.tight_layout()
save_fig(fig, "01_confusion_matrices.png")
plt.show()

# ── 7b. ROC Curves ──────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
ax.plot([0, 1], [0, 1], color="#AAAAAA", linestyle="--", linewidth=1, label="Random")

for model_name, (fpr, tpr, auc) in roc_data.items():
    ax.plot(fpr, tpr, color=PALETTE[model_name], linewidth=2,
            label=f"{model_name}  (AUC = {auc:.3f})")

style_ax(ax, title="ROC Curves — Test Set",
         xlabel="False Positive Rate", ylabel="True Positive Rate (Recall)")
ax.legend(fontsize=10, loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
save_fig(fig, "02_roc_curves.png")
plt.show()

# ── 7c. Calibration Curves + Brier Scores ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
ax.plot([0, 1], [0, 1], color="#AAAAAA", linestyle="--", linewidth=1, label="Perfectly calibrated")

for model_name, estimator in best_estimators.items():
    y_prob  = estimator.predict_proba(X_test_sel)[:, 1]
    brier   = brier_score_loss(y_test, y_prob)
    frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy="uniform")
    ax.plot(mean_pred, frac_pos, marker="o", linewidth=2,
            color=PALETTE[model_name],
            label=f"{model_name}  (Brier = {brier:.4f})")

style_ax(ax, title="Calibration Curves — Test Set",
         xlabel="Mean Predicted Probability", ylabel="Fraction of Positives")
ax.legend(fontsize=10)
fig.tight_layout()
save_fig(fig, "03_calibration_curves.png")
plt.show()

# ── 7d. Bar chart các chỉ số Metric ──────────────────────────────────────
metric_cols  = ["Recall", "ROC-AUC", "F1-Score", "Precision", "Accuracy"]
x            = np.arange(len(metric_cols))
bar_width    = 0.25
model_names  = metrics_df["Model"].tolist()

fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
for i, mn in enumerate(model_names):
    row    = metrics_df[metrics_df["Model"] == mn].iloc[0]
    values = [row[m] for m in metric_cols]
    bars   = ax.bar(x + i * bar_width, values, bar_width,
                    color=PALETTE[mn], edgecolor="white",
                    linewidth=0.7, label=mn, alpha=0.9)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x + bar_width)
ax.set_xticklabels(metric_cols, fontsize=11)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=10)
style_ax(ax, title="Model Comparison — Key Metrics (Test Set)", ylabel="Score")
ax.axvline(x=-0.15, color="#D94A4A", linestyle=":", linewidth=1.5, alpha=0.6)
fig.tight_layout()
save_fig(fig, "04_metric_comparison.png")
plt.show()

# ── 7e. Random Forest — Top feature importances ───────────────────────────
if "Random Forest" in best_estimators:
    rf_best   = best_estimators["Random Forest"]
    importances = pd.Series(rf_best.feature_importances_, index=selected_features)
    importances = importances.sort_values(ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(9, 6), facecolor=BG)
    ax.barh(importances.index, importances.values,
            color=PALETTE["Random Forest"], edgecolor="white", alpha=0.85)
    style_ax(ax, title="Random Forest — Top 15 Feature Importances",
             xlabel="Importance Score")
    fig.tight_layout()
    save_fig(fig, "05_rf_feature_importance.png")
    plt.show()

# ── 7f. XGBoost — Top feature importances ────────────────────────────────
if "XGBoost" in best_estimators:
    xgb_best    = best_estimators["XGBoost"]
    importances = pd.Series(xgb_best.feature_importances_, index=selected_features)
    importances = importances.sort_values(ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(9, 6), facecolor=BG)
    ax.barh(importances.index, importances.values,
            color=PALETTE["XGBoost"], edgecolor="white", alpha=0.85)
    style_ax(ax, title="XGBoost — Top 15 Feature Importances",
             xlabel="Importance Score")
    fig.tight_layout()
    save_fig(fig, "06_xgb_feature_importance.png")
    plt.show()



  STEP 7 — GENERATING EVALUATION PLOTS

  [Saved] outputs\model_results\01_confusion_matrices.png
  [Saved] outputs\model_results\02_roc_curves.png
  [Saved] outputs\model_results\03_calibration_curves.png
  [Saved] outputs\model_results\04_metric_comparison.png
  [Saved] outputs\model_results\05_rf_feature_importance.png
  [Saved] outputs\model_results\06_xgb_feature_importance.png


## Tổng kết (Final Summary)

In [17]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("\n  FINAL SUMMARY")
print("\n" + "=" * 70)
print(f"\n  Models trained           : {', '.join(MODELS.keys())}")
print(f"\n  Features used            : {len(selected_features)}")
print(f"\n  Champion model           : {best_name}")
print(f"\n  Champion Recall          : {best_row['Recall']:.4f}")
print(f"\n  Champion AUC             : {best_row['ROC-AUC']:.4f}")
print(f"\n  Saved models:")
for mname in MODELS:
    safe = mname.lower().replace(" ", "_")
    print(f"\n    -> models/best_{safe}.pkl")
print(f"\n    -> {champion_path}  (champion)")
print(f"\n    -> {feature_list_path}  (selected features)")
print(f"\n  Evaluation plots  : {os.path.abspath(RESULT_DIR)}/")
print(f"\n  Metrics CSV       : {metrics_csv}")
print("\n" + "=" * 70 + "\n")



  FINAL SUMMARY


  Models trained           : Logistic Regression, Random Forest, XGBoost

  Features used            : 20

  Champion model           : XGBoost

  Champion Recall          : 0.9606

  Champion AUC             : 0.9807

  Saved models:

    -> models/best_logistic_regression.pkl

    -> models/best_random_forest.pkl

    -> models/best_xgboost.pkl

    -> models\champion_model.pkl  (champion)

    -> models\selected_features.json  (selected features)

  Evaluation plots  : d:\Working\Coding\ml-project\outputs\model_results/

  Metrics CSV       : outputs\model_results\model_comparison.csv


